In [2]:
import sys
import os

# Move up one directory level from the current notebook location
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

In [3]:

from ingest import load_faq_data


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [6]:
v1.shape

(384,)

In [7]:
doc  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
doc_vector = model.encode(doc)

In [8]:
v1.dot(doc_vector)

np.float32(0.32332397)

In [9]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [11]:
v2.dot(doc_vector)

np.float32(0.019730438)

In [10]:
# 0.32 vs 0.01 => the document is more relevant to the first question than the second one.

In [11]:
# Get all the documents from the FAQ data
documents = load_faq_data()

In [12]:
# Create a list of texts by concatenating the question and answer for each document
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [13]:
len(texts)

1350

In [14]:
from tqdm.auto import tqdm


In [15]:
batch_size = 50
vectors = []

# Encode the texts in batches to avoid memory issues - We have made all the documents vectorized 
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [16]:
# If our question is q1 and vectorized question is v1, 
# we can find the most relevant document by calculating the dot product between v1 and all the document vectors. 
# The document with the highest dot product is the most relevant one.   

# scores = [v1.dot(doc_vector) for doc_vector in vectors]

# Better way instead of the above vector vector mulitiplication is to convert the vectors into a matrix
# and then do a matrix multiplication with the question vector.
import numpy as np
X = np.array(vectors)
scores = X.dot(v1)

# Select the most relevant document by finding the index of the highest score
most_relevant_doc_index = np.argmax(scores)
print(f"Most relevant document index: {most_relevant_doc_index}")
documents[most_relevant_doc_index]

# Select the top 5 documents with the highest scores
# -top_k selects the last top_k indices from the sorted scores, and [::-1] reverses the order to get the highest scores first.
top_k = 5
top_k_indices = np.argsort(scores)[-top_k:][::-1] # Alternatively: np.argsort(-scores)[:top_k]
for i, idx in enumerate(top_k_indices):
    print(f"Rank {i+1}: Document index: {idx}, Score: {scores[idx]}")
    print(documents[idx])
    print()

Most relevant document index: 2
Rank 1: Document index: 2, Score: 0.7629410028457642
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

Rank 2: Document index: 625, Score: 0.7579370737075806
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the 

In [17]:
from minsearch import VectorSearch  

vIndex =  VectorSearch(keyword_fields=['course'])
vIndex.fit(X, documents)

In [18]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vIndex.search(query_vector, num_results=5)

In [19]:
for result in results:
    print(f"{result}")
    print()

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Relate

In [20]:
results = vIndex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

for result in results:
    print(f"{result}")
    print()

{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

{'id': '69d122f12e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?', 'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}

{'id': 'bd31146b0e', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'When will the course be offered next?', '

In [21]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [22]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [23]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [24]:
# This uses keyword search - text based
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.'

In [25]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [26]:
vector_assistant = RAGVector(
    embedder=model,
    index=vIndex,
    llm_client=openai_client,
)

In [27]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even after the program has begun. You can start learning and submit homework while the form is open. If you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [28]:
# Pain points for using Vector Search:
# 1. Have to embed each document and build index
# 2. Everything happens in memory - if the dataset is large, it can be slow and memory intensive
# 3. Brute Force Search: For every query we compare the query vector against every single document. This is not scalable for large datasets. 
# We need to use approximate nearest neighbor search algorithms for effecient vector search.

# We can use sqlitesearch for approximate nearest neighbor search. It stores vectors in SQLite, a real on-disk database, and uses ANN strategies for retrieval. 

In [29]:
from sqlitesearch import VectorSearchIndex


In [30]:
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [31]:
vs_index.fit(vectors, documents)

In [32]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [34]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [35]:
# from sentence_transformers import SentenceTransformer
# from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [37]:
results

[{'id': '5ca6890c1a',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: How to run producer/consumer/kstreams/etc in terminal',
  'answer': 'In the project directory, run:\n\n```bash\njava -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java\n```'},
 {'id': 'cd8a62fc55',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 7: Streaming',
  'question': 'Java Kafka: When running the producer/consumer/etc java scripts, no results retrieved or no message sent',
  'answer': 'For example, when running `JsonConsumer.java`, you might see:\n\n```\nConsuming form kafka started\n\nRESULTS:::0\n\nRESULTS:::0\n\nRESULTS:::0\n```\n\nOr when running `JsonProducer.java`, you might encounter:\n\n```\nException in thread "main" java.util.concurrent.ExecutionException: org.apache.kafka.common.errors.SaslAuthenticationException: Authentication failed\n```\n\n**Solution:**\n\n1. Ensure the `StreamsConfig.BO

In [38]:
from rag_helper import RAGBase
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [39]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project while submissions are still being accepted.'